# G31-04 Stock-Out Risk Prediction: data preparation
Group G31-04, Dataset S1 (an online shop, 3 May to 18 Sep 2015). Data Engineer part, Progress 1.

**Goal of this notebook.** Turn the 4 raw files into one clean, model-ready table: one row = one in-stock, active item in
one week, with features from the past and the target *out of stock next week*. Every step shows what it does, why,
and how many rows and columns there are before and after.

| Section | What happens |
|---|---|
| 0 | Setup and the 4 raw files |
| 1 | Load the raw data and look at it |
| 2 | Data quality checks (what is wrong or unusual, and what we decide) |
| 3 | Cleaning |
| 4 | Weekly item table and the target |
| 5 | Feature engineering, with a leakage check |
| 6 | Final dataset and data dictionary |
| 7 | Pictures of the data, the data pipeline tree, and initial EDA |
| 8 | Save everything |

**How to run.** Google Colab: *Runtime > Run all* (about 5 to 10 minutes, about 4 GB RAM). Jupyter works too.
In cell 0.2 choose where the raw files come from. All results go to the folder `G31-04_run` and a zip.

## 0. Setup

In [ ]:
#@title 0.1 Libraries and folders
import os, sys, glob, shutil, zipfile, subprocess, importlib.util, textwrap
from pathlib import Path

for lib in ["pandas", "numpy", "matplotlib", "openpyxl", "gdown"]:
    if importlib.util.find_spec(lib) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", lib], check=True)
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

IN_COLAB = "google.colab" in sys.modules
RUN_DIR = Path(os.environ.get("NB_RUN_DIR", "/content/G31-04_run" if IN_COLAB else "G31-04_run")).resolve()
RAW_DIR, OUT_DIR, CHART_DIR = RUN_DIR / "data_raw", RUN_DIR / "outputs", RUN_DIR / "outputs" / "charts"
for d in (RAW_DIR, OUT_DIR, CHART_DIR):
    d.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
DAY = 86_400_000          # one day in milliseconds (all timestamps in the files are Unix milliseconds)
print("run folder:", RUN_DIR)
print("python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "| matplotlib", matplotlib.__version__)

In [ ]:
#@title 0.2 Get the 4 raw CSV files
#@markdown **USE_MY_DRIVE** (Colab): if your Google Drive has `DRIVE_FOLDER` with the sub-folders `G31-04-RAW` (the 4 CSVs) and
#@markdown `G31-04-OUTPUT`, the raw files are read from there and the results are saved there. Without that folder nothing is
#@markdown read from or written to your Drive, and `RAW_SOURCE` is used: `download` (public link, no sign-in), `upload`
#@markdown (pick the files or one zip), or `folder` (a path in `RAW_FOLDER`).
USE_MY_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "/content/drive/MyDrive/Colab Notebooks/G31-04"  #@param {type:"string"}
RAW_SOURCE = "download"  #@param ["download", "upload", "folder"]
DOWNLOAD_LINK = "https://drive.google.com/drive/folders/1L2N2pFDTrI8cf2VQQMiS9PuqRIN0EuIq"  #@param {type:"string"}
RAW_FOLDER = ""  #@param {type:"string"}
# outside Colab (automatic tests) these can be set as environment variables
RAW_SOURCE = os.environ.get("NB_RAW_SOURCE", RAW_SOURCE)
RAW_FOLDER = os.environ.get("NB_RAW_FOLDER", RAW_FOLDER)
DRIVE_FOLDER = os.environ.get("NB_DRIVE_FOLDER", DRIVE_FOLDER)
DRIVE_RAW, DRIVE_OUT = Path(DRIVE_FOLDER) / "G31-04-RAW", Path(DRIVE_FOLDER) / "G31-04-OUTPUT"

FILES = ["item_properties_part1.csv", "item_properties_part2.csv", "events.csv", "category_tree.csv"]
have = lambda: [f for f in FILES if (RAW_DIR / f).exists()]


def collect(src):
    """Copy the 4 CSVs from a folder (also from sub-folders) into data_raw/, or take them out of a zip there."""
    for name in FILES:
        if (RAW_DIR / name).exists():
            continue
        hits = glob.glob(f"{src}/**/{name}", recursive=True)
        if hits:
            shutil.copy(hits[0], RAW_DIR / name)
            continue
        for z in glob.glob(f"{src}/**/*.zip", recursive=True):
            with zipfile.ZipFile(z) as zf:
                member = next((m for m in zf.namelist() if m.endswith(name)), None)
                if member:
                    with zf.open(member) as s, open(RAW_DIR / name, "wb") as d:
                        shutil.copyfileobj(s, d)
                    break


if USE_MY_DRIVE and IN_COLAB and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Google Drive not connected ({type(e).__name__}).")
USE_MY_DRIVE = USE_MY_DRIVE and DRIVE_RAW.is_dir()
if USE_MY_DRIVE:
    print("raw files from", DRIVE_RAW)
    collect(DRIVE_RAW)
if len(have()) < len(FILES):
    print("raw files: RAW_SOURCE =", RAW_SOURCE)
    if RAW_SOURCE == "download":
        import gdown
        tmp = RUN_DIR / "download"
        if "/folders/" in DOWNLOAD_LINK:
            gdown.download_folder(url=DOWNLOAD_LINK, output=str(tmp), quiet=False, use_cookies=False)
        else:
            tmp.mkdir(exist_ok=True)
            gdown.download(url=DOWNLOAD_LINK, output=str(tmp / "raw.zip"), quiet=False, fuzzy=True)
        collect(tmp)
    elif RAW_SOURCE == "upload":
        if not IN_COLAB:
            raise RuntimeError(f"Upload works in Colab only. Put the 4 files into {RAW_DIR} and run again.")
        from google.colab import files
        up = RUN_DIR / "upload"
        up.mkdir(exist_ok=True)
        for name, data in files.upload().items():
            (up / name).write_bytes(data)
        collect(up)
    else:
        collect(RAW_FOLDER)
missing = [f for f in FILES if f not in have()]
if missing:
    raise FileNotFoundError(f"Missing raw files: {missing}. Check the settings in this cell.")
for f in FILES:
    print(f"{f:28s} {(RAW_DIR / f).stat().st_size / 1e6:9.1f} MB")

In [ ]:
#@title 0.3 Process log: every step records its input table, its output table, and rows and columns before and after
CLEAN_LOG = []


def log_step(step, action, before, after, src=None, dst=None, note="", funnel=None):
    """step: the cell number. before / after: a DataFrame or a (rows, columns) pair.
    src -> dst: the table(s) the step reads and the table it writes (src may be a list). When dst is also in src,
    the step changes that table in place. These records give the cleaning log and the data pipeline tree (section 7).
    funnel: a short label if the step belongs in the row-count chart."""
    sh = lambda x: x.shape if hasattr(x, "shape") else x
    (rb, cb), (ra, ca) = sh(before), sh(after)
    src = [] if src is None else [src] if isinstance(src, str) else list(src)
    CLEAN_LOG.append(dict(step=step, action=action, input=" + ".join(src), output=dst or "", rows_before=int(rb), cols_before=int(cb),
                          rows_after=int(ra), cols_after=int(ca), rows_changed=int(ra) - int(rb), note=note, funnel=funnel, _src=src))
    print(f"[{step}] {action}: {int(rb):,} x {cb} -> {int(ra):,} x {ca}" + (f"   ({note})" if note else ""))

## 1. Load the raw data and look at it
The files have no description, so we first look at their columns and rows. `item_properties` is large (about 900 MB),
so it is read in parts of 2 million rows. From it we keep only the two properties we can read and need
(`available` and `categoryid`); section 1.1 shows why the others cannot be used.

In [ ]:
#@title 1.1 item_properties (part 1 and part 2): read in chunks
ip_head = pd.read_csv(RAW_DIR / "item_properties_part1.csv", nrows=5, dtype=str)
ip2_head = pd.read_csv(RAW_DIR / "item_properties_part2.csv", nrows=5, dtype=str)
print("Columns and the first 5 rows of item_properties_part1.csv:")
display(ip_head)
print("Columns and the first 5 rows of item_properties_part2.csv (the same 4 columns; part 2 continues part 1):")
display(ip2_head)

KEEP = ["available", "categoryid"]
kept, prop_rows, n_rows = [], pd.Series(dtype="int64"), {}
for name in ["item_properties_part1.csv", "item_properties_part2.csv"]:
    n = 0
    for ch in pd.read_csv(RAW_DIR / name, chunksize=2_000_000, dtype={"timestamp": "int64", "itemid": "int64", "property": str, "value": str}):
        n += len(ch)
        prop_rows = prop_rows.add(ch["property"].value_counts(), fill_value=0)
        kept.append(ch[ch["property"].isin(KEEP)])
    n_rows[name] = n
    log_step("1.1", f"read {name}", (0, 0), (n, 4), dst=name)
ip_rows = sum(n_rows.values())
props = pd.concat(kept, ignore_index=True)
del kept
log_step("1.1", "stack part 1 and part 2", (ip_rows, 4), (ip_rows, 4), src=list(n_rows), dst="item_properties",
         note="same 4 columns", funnel="Raw item properties")

print(f"\n{ip_rows:,} rows, {len(prop_rows):,} different property names.")
print("Most frequent property names (most names are numbers, their values are coded text):")
display(prop_rows.sort_values(ascending=False).head(12).astype(int).rename("rows").to_frame())
print("The two properties we use:")
display(prop_rows.reindex(KEEP).astype(int).rename("rows").to_frame())

In [ ]:
#@title 1.2 events and category_tree
ev = pd.read_csv(RAW_DIR / "events.csv")
n_events_raw = len(ev)
log_step("1.2", "read events.csv", (0, 0), ev, dst="events.csv")
tree = pd.read_csv(RAW_DIR / "category_tree.csv")
log_step("1.2", "read category_tree.csv", (0, 0), tree, dst="category_tree.csv")
for name, df in [("events", ev), ("category_tree", tree)]:
    print(f"\n{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    display(df.head(5))
    display(pd.DataFrame({"type": df.dtypes.astype(str), "empty values": df.isna().sum(), "distinct values": df.nunique()}))

In [ ]:
#@title 1.3 What one row means, and the time range
fmt = lambda ms: pd.to_datetime(ms, unit="ms").strftime("%d %b %Y %H:%M UTC")
display(pd.DataFrame([
    ("item_properties", "one value of one property of one item at one time", ip_rows, fmt(props["timestamp"].min()), fmt(props["timestamp"].max())),
    ("events", "one customer action (view, addtocart, transaction) on one item", len(ev), fmt(ev["timestamp"].min()), fmt(ev["timestamp"].max())),
    ("category_tree", "one category and its parent category", len(tree), "", ""),
], columns=["file", "one row is", "rows", "first time", "last time"]))
print("Event types:")
display(ev["event"].value_counts().rename("rows").to_frame())

## 2. Data quality checks
Before changing anything we count what is wrong or unusual. The last column says what we do about it in section 3 or 4.

In [ ]:
#@title 2.1 Run all checks
av_raw = props[props["property"] == "available"]
cat_raw = props[props["property"] == "categoryid"]
av_num = pd.to_numeric(av_raw["value"], errors="coerce")
cat_num = pd.to_numeric(cat_raw["value"], errors="coerce")
snap_days = pd.to_datetime(av_raw["timestamp"].unique(), unit="ms").sort_values()
gaps = pd.Series(snap_days[1:] - snap_days[:-1]).dt.days
missing_weeks = [(snap_days[i] + pd.Timedelta(days=7)).strftime("%d %b") for i in range(len(gaps)) if gaps.iloc[i] > 7]
ev_items, av_items = set(ev["itemid"]), set(av_raw["itemid"])
no_status = ev_items - av_items
per_visitor = ev.groupby("visitorid").size()
heavy = per_visitor[per_visitor >= 1000].index
tx_mismatch = int(((ev["event"] == "transaction") != ev["transactionid"].notna()).sum())
tree_ids = set(tree["categoryid"])
item_cats = set(cat_num.dropna().astype(int))
never_in_stock = int((av_num.groupby(av_raw["itemid"]).max() == 0).sum())

checks = [
    ("item_properties: property names that are coded numbers", len(prop_rows) - len(KEEP),
     "not used: names and values cannot be read; we keep available and categoryid"),
    ("available: values other than 0 or 1", int((~av_num.isin([0, 1])).sum()), "drop such rows (section 3)"),
    ("available: same item and same time recorded twice", int(av_raw.duplicated(["itemid", "timestamp"]).sum()), "keep the last one"),
    ("categoryid: values that are not a number", int(cat_num.isna().sum()), "drop such rows"),
    ("available: weekly snapshot dates", len(snap_days), "these are the weeks of our table"),
    ("available: weekly snapshot missing", len(missing_weeks), f"missing: {', '.join(missing_weeks) or 'none'}; not invented, the week before gets no target"),
    ("available: items never in stock in any record", never_in_stock, "kept; they are never in stock, so they never enter the final table"),
    ("events: exact duplicate rows (all 5 columns equal)", int(ev.duplicated().sum()), "remove (section 3)"),
    ("events: transactionid present without a purchase, or missing for a purchase", tx_mismatch, "check only; transactionid is not used"),
    ("events: items with events but no stock record", len(no_status), "kept in the events, but they cannot get a stock status or a target"),
    ("events: rows on those items", int(ev["itemid"].isin(no_status).sum()), "same as above"),
    ("events: visitors with 1,000 or more events", len(heavy),
     f"kept; they made {int(ev['visitorid'].isin(heavy).sum()):,} events and {int((ev['visitorid'].isin(heavy) & (ev['event'] == 'transaction')).sum()):,} purchases"),
    ("category_tree: categories without a parent (top level)", int(tree["parentid"].isna().sum()), "normal; the parent of a top-level category is the category itself"),
    ("category_tree: category ids listed twice", int(tree["categoryid"].duplicated().sum()), "check only"),
    ("category ids of items that are not in category_tree", len(item_cats - tree_ids), 'parent category = "Unknown"'),
    ("item-weeks without a known category", None, 'category = "Unknown" plus a flag (counted in step 4.1, where this table is shown again)'),
]
quality = pd.DataFrame(checks, columns=["check", "count", "what we do"]).astype({"count": "Int64"})
pd.set_option("display.max_colwidth", 120)
quality

## 3. Cleaning
Each step below changes the data and writes its row and column counts into the cleaning log.

In [ ]:
#@title 3.1 Keep the two readable properties, then split them
log_step("3.1", "keep property in {available, categoryid}", (ip_rows, 4), props, src="item_properties", dst="kept properties",
         note=f"{len(prop_rows) - len(KEEP):,} other properties dropped", funnel="Keep available and categoryid")
av = props[props["property"] == "available"].drop(columns="property")
cat = props[props["property"] == "categoryid"].drop(columns="property")
log_step("3.1", "split: available rows", props, av, src="kept properties", dst="available", note="property column dropped")
log_step("3.1", "split: categoryid rows", props, cat, src="kept properties", dst="categoryid", note="property column dropped")
del props

In [ ]:
#@title 3.2 Types: text values to numbers; drop values that are not valid
n0 = av.shape
av = av.assign(value=pd.to_numeric(av["value"], errors="coerce"))
av = av[av["value"].isin([0, 1])].astype({"value": "int8"})
log_step("3.2", "available: value to 0/1, drop other values", n0, av, src="available", dst="available")
n0 = cat.shape
cat = cat.assign(value=pd.to_numeric(cat["value"], errors="coerce")).dropna(subset=["value"]).astype({"value": "int64"})
log_step("3.2", "categoryid: value to a whole number, drop other values", n0, cat, src="categoryid", dst="categoryid")

In [ ]:
#@title 3.3 Duplicates
n0 = av.shape
av = av.sort_values(["itemid", "timestamp"]).drop_duplicates(["itemid", "timestamp"], keep="last")
log_step("3.3", "available: one record per item and time (keep the last)", n0, av, src="available", dst="available")
n0 = cat.shape
cat = cat.sort_values(["itemid", "timestamp"]).drop_duplicates(["itemid", "timestamp"], keep="last")
log_step("3.3", "categoryid: one record per item and time (keep the last)", n0, cat, src="categoryid", dst="categoryid")
n0 = ev.shape
ev = ev.drop_duplicates(ignore_index=True)
log_step("3.3", "events: drop exact duplicate rows", n0, ev, src="events.csv", dst="events (clean)", note=f"{n0[0] - len(ev):,} removed")
prev = av.groupby("itemid")["value"].shift()
print(f"\nThe stock status is repeated every week even without a change: of {len(av):,} available rows, "
      f"{int(prev.isna().sum()):,} are the first record of an item and {int((prev.notna() & (prev != av['value'])).sum()):,} are a change.")

In [ ]:
#@title 3.4 Time: snapshot dates, and events sorted by time
SNAPS = np.sort(av["timestamp"].unique())
DATES = pd.to_datetime(SNAPS, unit="ms").strftime("%Y-%m-%d")
step_days = np.diff(SNAPS) / DAY
print(f"{len(SNAPS)} snapshots, from {DATES[0]} to {DATES[-1]}; days between snapshots: "
      + ", ".join(f"{int(d)} days ({int((step_days == d).sum())} times)" for d in sorted(set(step_days))))
print("Each snapshot is one exact moment:", pd.to_datetime(SNAPS[:3], unit="ms").strftime("%a %d %b %H:%M UTC").tolist(), "...")
ev = ev.sort_values("timestamp", ignore_index=True)
EV_T = ev["timestamp"].to_numpy()
ev["event"] = ev["event"].astype("category")
print("Timestamps are kept in UTC. All windows below are counted back from the exact snapshot time.")

## 4. Weekly item table and the target
**Definitions** (from the proposal):
- *Snapshot t*: one of the weekly moments when `available` was recorded.
- *Status at t*: the last known `available` value at or before t (carried forward). 1 = in stock, 0 = out of stock.
- *Active at t*: at least one event (view, add-to-cart or purchase) in the 7 days up to t.
- *Target* `y_next_out`: 1 if the item is out of stock at the snapshot exactly 7 days after t. If that snapshot does not
  exist (the week before the missing week, and the last week), the week has no target and is not used.
- *Rows we keep*: items that are **in stock and active** at t. An item that is already out of stock needs no prediction.
- *Category at t*: the last known `categoryid` at or before t. None yet: "Unknown" (code -1). We never fill backwards
  from a later record, because that would use the future.

In [ ]:
#@title 4.1 Status and category per item and snapshot (items x snapshots)
S = av.pivot(index="itemid", columns="timestamp", values="value").reindex(columns=SNAPS).astype("float32")
log_step("4.1", "status matrix (rows = items, columns = snapshots)", av, S, src="available", dst="status matrix",
         note=f"{int(S.isna().sum().sum()):,} empty cells")
S = S.ffill(axis=1)
ITEMS = S.index
SV = S.to_numpy()
known = ~np.isnan(SV)
log_step("4.1", "carry the last status forward; count item-weeks with a known status", (S.size, 3), (int(known.sum()), 3),
         src="status matrix", dst="item-weeks with a status",
         note=f"{len(ITEMS):,} items x {len(SNAPS)} snapshots = {S.size:,}; {S.size - int(known.sum()):,} are before the first status",
         funnel="One row per item and week")
C = cat.pivot(index="itemid", columns="timestamp", values="value").astype("float64")
C = C.reindex(columns=sorted(set(C.columns) | set(SNAPS))).ffill(axis=1)[SNAPS].reindex(ITEMS)
n_unknown = int(C.isna().sum().sum())
CV = C.fillna(-1).to_numpy().astype("int64")
log_step("4.1", 'category per item and snapshot; none yet -> "Unknown" (-1)', cat, C, src="categoryid",
         dst="category per item-week", note=f"{n_unknown:,} item-weeks have no category yet")
PARENT = {int(c): int(c if pd.isna(p) else p) for c, p in zip(tree["categoryid"], tree["parentid"])}
quality.loc[quality["check"].str.startswith("item-weeks without a known category"), "count"] = n_unknown
print("\nData quality checks, now with the count from this step:")
display(quality)

In [ ]:
#@title 4.2 Helper: events in a time window before a snapshot
def window(t, days):
    """Events with t - days < time <= t (uses only the past)."""
    lo, hi = np.searchsorted(EV_T, t - days * DAY, side="right"), np.searchsorted(EV_T, t, side="right")
    return ev.iloc[lo:hi]


def counts(w):
    c = w.groupby(["itemid", "event"], observed=False).size().unstack(fill_value=0)
    return c.reindex(columns=["view", "addtocart", "transaction"], fill_value=0)

## 5. Feature engineering
All features are built from data **at or before the snapshot time t**; the target comes from t + 7 days.

| Group | Features | Why |
|---|---|---|
| Demand level | views, add-to-carts, purchases in the last 1 and 4 weeks; `demand_1w`, `demand_4w` (views + add-to-carts) | Popular items sell more units, so they can run out faster |
| Demand change | `demand_prev3w_avg`, `demand_trend` | A sudden rise can empty the stock before it is refilled |
| Buying intent | `atc_rate_1w` (add-to-carts per view), `buy_rate_4w` (purchases per view) | Intent and sales take units out of stock, views alone do not |
| Breadth | `visitors_1w` (different visitors) | Many people vs one person clicking many times |
| Recency of sales | `days_since_last_tx`, `has_tx_before` | An item that sold recently is more likely to run out |
| Stock history | `changes_4w`, `snapshots_in_status`, `oos_share_4w` | Items that switched status recently tend to switch again |
| Category | `cat`, `cat_missing`, `parent_cat`, `cat_in_tree`, `cat_demand_1w`, `item_share_of_cat_demand` | Risk differs by category; a busy category or a main seller in it behaves differently |

In [ ]:
#@title 5.1 Build the rows and the features, week by week
# stock history over all snapshots
RUN = np.zeros(SV.shape, dtype="int16")      # consecutive snapshots with the same status, up to and including t
RUN[:, 0] = known[:, 0]
for j in range(1, len(SNAPS)):
    same = known[:, j] & known[:, j - 1] & (SV[:, j] == SV[:, j - 1])
    RUN[:, j] = np.where(known[:, j], np.where(same, RUN[:, j - 1] + 1, 1), 0)

tx_all = ev[ev["event"] == "transaction"]
TX_T = tx_all["timestamp"].to_numpy()
rows, n_weeks, n_label, n_instock, n_active = [], 0, 0, 0, 0
for j, t in enumerate(SNAPS):
    has_label = j + 1 < len(SNAPS) and SNAPS[j + 1] - t == 7 * DAY
    kn = known[:, j]
    if not has_label:
        continue
    n_weeks += 1
    n_label += int(kn.sum())
    instock = kn & (SV[:, j] == 1)
    n_instock += int(instock.sum())
    w1, w4 = window(t, 7), window(t, 28)
    c1, c4 = counts(w1).reindex(ITEMS, fill_value=0), counts(w4).reindex(ITEMS, fill_value=0)
    active = (c1.sum(axis=1).to_numpy() > 0)
    keep = instock & active
    n_active += int(keep.sum())
    d = pd.DataFrame(index=ITEMS[keep])
    d["snapshot"] = DATES[j]
    d["views_1w"], d["atc_1w"], d["tx_1w"] = (c1.loc[d.index, k].to_numpy() for k in ["view", "addtocart", "transaction"])
    d["views_4w"], d["atc_4w"], d["tx_4w"] = (c4.loc[d.index, k].to_numpy() for k in ["view", "addtocart", "transaction"])
    vis = w1[w1["event"].isin(["view", "addtocart"])].groupby("itemid")["visitorid"].nunique()
    d["visitors_1w"] = vis.reindex(d.index, fill_value=0).to_numpy()
    last_tx = tx_all.iloc[: np.searchsorted(TX_T, t, side="right")].groupby("itemid")["timestamp"].max()
    d["days_since_last_tx"] = ((t - last_tx.reindex(d.index)) / DAY).round(2).to_numpy()
    lo = max(0, j - 4)
    seg = SV[keep, lo:j + 1]
    d["changes_4w"] = (np.abs(np.diff(seg, axis=1)) > 0).sum(axis=1) if seg.shape[1] > 1 else 0
    d["snapshots_in_status"] = RUN[keep, j].astype(int)
    prevs = SV[keep, lo:j]                    # the up to 4 snapshots before t
    n_known_prev = (~np.isnan(prevs)).sum(axis=1)
    d["oos_share_4w"] = np.where(n_known_prev > 0, (prevs == 0).sum(axis=1) / np.maximum(n_known_prev, 1), np.nan)
    d["cat"] = CV[keep, j]
    # category demand: all active items of the category at t (in stock or not)
    all_active = pd.Series(c1["view"].to_numpy() + c1["addtocart"].to_numpy(), index=ITEMS)[active]
    cat_dem = all_active.groupby(CV[active, j]).sum()
    d["cat_demand_1w"] = cat_dem.reindex(d["cat"]).to_numpy()
    d["y_next_out"] = (SV[keep, j + 1] == 0).astype("int8")
    rows.append(d)
df = pd.concat(rows).rename_axis("itemid").reset_index()
del rows, c1, c4, w1, w4

total = len(ITEMS) * len(SNAPS)
log_step("5.1", "keep weeks that have a target (next snapshot exactly 7 days later)", (int(known.sum()), 3), (n_label, 3),
         src="item-weeks with a status", dst="weeks with a target", note=f"{len(SNAPS) - n_weeks} snapshots without a next week",
         funnel="Weeks with a target")
log_step("5.1", "keep items in stock at t", (n_label, 3), (n_instock, 3), src="weeks with a target", dst="in stock at t",
         funnel="In stock at t")
log_step("5.1", "keep items active at t (an event in the last 7 days)", (n_instock, 3), (n_active, 3),
         src=["in stock at t", "events (clean)"], dst="in stock and active", funnel="Active at t (final rows)")
log_step("5.1", "build the target and the event, stock and category features", (n_active, 3), df,
         src=["in stock and active", "events (clean)", "category per item-week"], dst="final table")

In [ ]:
#@title 5.2 Features computed from the columns above, and the category fields
df["demand_1w"] = df["views_1w"] + df["atc_1w"]
df["demand_4w"] = df["views_4w"] + df["atc_4w"]
df["demand_prev3w_avg"] = (df["demand_4w"] - df["demand_1w"]) / 3
df["demand_trend"] = (df["demand_1w"] + 1) / (df["demand_prev3w_avg"] + 1)
df["atc_rate_1w"] = np.where(df["views_1w"] > 0, df["atc_1w"] / df["views_1w"].where(df["views_1w"] > 0), 0.0)
df["buy_rate_4w"] = np.where(df["views_4w"] > 0, df["tx_4w"] / df["views_4w"].where(df["views_4w"] > 0), 0.0)
df["has_tx_before"] = df["days_since_last_tx"].notna().astype("int8")
df["cat_missing"] = (df["cat"] == -1).astype("int8")
df["cat_in_tree"] = df["cat"].isin(list(PARENT)).astype("int8")
df["parent_cat"] = df["cat"].map(PARENT).fillna(-1).astype("int64")
df.loc[df["cat"] == -1, "cat_demand_1w"] = np.nan              # "Unknown" is not a real category
df["item_share_of_cat_demand"] = np.where(df["cat_demand_1w"] > 0, df["demand_1w"] / df["cat_demand_1w"], np.nan)
weeks = sorted(df["snapshot"].unique())
SPLIT = {w: ("train" if i < 10 else "validate" if i < 13 else "test") for i, w in enumerate(weeks)}
df["split"] = df["snapshot"].map(SPLIT)
print("split by time (never random):")
display(df.groupby("split", sort=False)["snapshot"].agg(["min", "max", "nunique", "size"]).rename(
    columns={"min": "first week", "max": "last week", "nunique": "weeks", "size": "rows"}))
log_step("5.2", "add the computed features, parent category and split", (len(df), 3), df,
         src=["final table", "category_tree.csv"], dst="final table")

In [ ]:
#@title 5.3 Leakage and correctness check: recompute 300 random rows straight from the cleaned data
rng = np.random.default_rng(0)
sample = df.sample(300, random_state=0)
bad = []
for r in sample.itertuples():
    t = SNAPS[list(DATES).index(r.snapshot)]
    e = ev[(ev["itemid"] == r.itemid)]
    v1 = int(((e["timestamp"] > t - 7 * DAY) & (e["timestamp"] <= t) & (e["event"] == "view")).sum())
    x4 = int(((e["timestamp"] > t - 28 * DAY) & (e["timestamp"] <= t) & (e["event"] == "transaction")).sum())
    tx = e.loc[(e["event"] == "transaction") & (e["timestamp"] <= t), "timestamp"]
    last = round((t - tx.max()) / DAY, 2) if len(tx) else np.nan
    a = av[av["itemid"] == r.itemid]
    now = a.loc[a["timestamp"] <= t, "value"].iloc[-1]
    nxt = a.loc[a["timestamp"] <= t + 7 * DAY, "value"].iloc[-1]
    ok = (v1 == r.views_1w and x4 == r.tx_4w and now == 1 and int(nxt == 0) == r.y_next_out
          and ((np.isnan(last) and np.isnan(r.days_since_last_tx)) or last == r.days_since_last_tx))
    if not ok:
        bad.append(r.Index)
print(f"rows checked: {len(sample)}, rows that differ: {len(bad)}")
assert not bad, "a feature or the target does not match a direct recount"
print("Every checked feature uses only events at or before t, and the target is the status 7 days later.")

## 6. Final dataset and data dictionary
Missing values that remain are *meaningful*, so they are not filled here: `days_since_last_tx` is empty when the item was
never bought before t (see `has_tx_before`), `cat_demand_1w` and `item_share_of_cat_demand` are empty for the "Unknown"
category, `oos_share_4w` is empty when there is no earlier snapshot. Each model fills them in its own way (trees can use
them as they are; Logistic Regression needs a fill value plus the flag).

In [ ]:
#@title 6.1 Column order and the data dictionary
DICT = {
    "itemid": ("integer", "Item id", "key"),
    "snapshot": ("date", "Snapshot date t (the week of the row)", "key, split by time"),
    "split": ("text", "train / validate / test, by date", "time-based evaluation"),
    "y_next_out": ("0/1", "1 = out of stock at the snapshot 7 days after t", "target"),
    "views_1w": ("integer", "Views in the 7 days up to t", "demand"),
    "atc_1w": ("integer", "Add-to-carts in the 7 days up to t", "buying intent"),
    "tx_1w": ("integer", "Purchases in the 7 days up to t", "sales take units out of stock"),
    "views_4w": ("integer", "Views in the 28 days up to t", "longer demand level"),
    "atc_4w": ("integer", "Add-to-carts in the 28 days up to t", "longer intent level"),
    "tx_4w": ("integer", "Purchases in the 28 days up to t", "longer sales level"),
    "demand_1w": ("integer", "views_1w + atc_1w", "demand as defined in the proposal"),
    "demand_4w": ("integer", "views_4w + atc_4w", "monthly demand"),
    "demand_prev3w_avg": ("number", "(demand_4w - demand_1w) / 3", "normal weekly demand before the last week"),
    "demand_trend": ("number", "(demand_1w + 1) / (demand_prev3w_avg + 1); above 1 = rising", "a sudden rise can empty the stock"),
    "atc_rate_1w": ("number", "atc_1w / views_1w (0 if no views)", "intent per view"),
    "buy_rate_4w": ("number", "tx_4w / views_4w (0 if no views)", "sales per view"),
    "visitors_1w": ("integer", "Different visitors with a view or add-to-cart in the 7 days up to t", "broad vs narrow demand"),
    "days_since_last_tx": ("number, can be empty", "Days from the last purchase at or before t; empty = never bought before t", "recent sales"),
    "has_tx_before": ("0/1", "1 = the item was bought at least once before t", "flag for the empty value above"),
    "changes_4w": ("integer", "Status changes over the last 4 snapshot steps", "items that switch tend to switch again"),
    "snapshots_in_status": ("integer", "Consecutive snapshots with the current status, up to t (only since the first record)", "how stable the stock is"),
    "oos_share_4w": ("number, can be empty", "Share of the up to 4 previous snapshots with status 0", "recent out-of-stock history"),
    "cat": ("integer", "Last known category at or before t; -1 = Unknown", "risk differs by category"),
    "cat_missing": ("0/1", "1 = category Unknown", "keeps Unknown apart from real codes"),
    "parent_cat": ("integer", "Parent category from category_tree; a top-level category is its own parent; -1 = Unknown", "groups small categories"),
    "cat_in_tree": ("0/1", "1 = cat is found in category_tree", "marks categories missing from the tree"),
    "cat_demand_1w": ("integer, can be empty", "demand_1w of all active items of the category at t (any status)", "how busy the category is"),
    "item_share_of_cat_demand": ("number, can be empty", "demand_1w / cat_demand_1w", "main seller or small item in its category"),
}
assert set(DICT) == set(df.columns), f"not described: {set(df.columns) - set(DICT)}; not in table: {set(DICT) - set(df.columns)}"
df = df[list(DICT)].sort_values(["snapshot", "itemid"], ignore_index=True)
first = df.iloc[0]
dictionary = pd.DataFrame([(c, t, m, first[c], w) for c, (t, m, w) in DICT.items()],
                          columns=["column", "type", "meaning", "example (row 1)", "why"])
log_step("6.1", "final column order, checked against the data dictionary", df, df, src="final table", dst="final table")
print(f"Final table: {len(df):,} rows x {df.shape[1]} columns | target = 1 in {int(df.y_next_out.sum()):,} rows ({100 * df.y_next_out.mean():.1f}%)")
dictionary

In [ ]:
#@title 6.2 The final table: first rows and a summary of every column
display(df.head(8))
summary = pd.DataFrame({"type": df.dtypes.astype(str), "empty": df.isna().sum(), "empty %": (100 * df.isna().mean()).round(1),
                        "min": df.min(numeric_only=True), "median": df.median(numeric_only=True), "max": df.max(numeric_only=True)})
summary

In [ ]:
#@title 6.3 Process log (all steps: input, output, rows and columns before and after)
clean_log = pd.DataFrame(CLEAN_LOG)
LOG_COLS = ["step", "action", "input", "output", "rows_before", "cols_before", "rows_after", "cols_after", "rows_changed", "note"]
clean_log[LOG_COLS]

## 7. Initial EDA
Charts are saved in `outputs/charts`. The titles say only what the data shows; the interpretation belongs in the report.

In [ ]:
#@title 7.1 Chart style and helpers
SURFACE, INK, INK2, GRID, BAND = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1", "#f0efec"
BLUE, ORANGE, RED, HEAD = "#2a78d6", "#eb6834", "#e34948", "#5f6a86"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                     "font.size": 11, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": GRID, "axes.axisbelow": True, "legend.frameon": False})


def figure(title, subtitle, left=0.1):
    fig, ax = plt.subplots(figsize=(10, 5.6), dpi=110)
    fig.subplots_adjust(left=left, right=0.95, top=0.77, bottom=0.16)
    fig.text(0.02, 0.955, title, fontsize=15, fontweight="bold", va="top")
    fig.text(0.02, 0.885, textwrap.fill(subtitle, 125), fontsize=9.5, color=INK2, va="top")
    return fig, ax


def finish(fig, name):
    fig.text(0.01, 0.01, "Group G31-04, Dataset S1", color=INK2, fontsize=8)
    fig.savefig(CHART_DIR / name, dpi=160)
    plt.show()


def table_picture(df_, title, subtitle, name, height=0.5):
    fig = plt.figure(figsize=(10, 5.6), dpi=110)
    fig.text(0.02, 0.955, title, fontsize=15, fontweight="bold", va="top")
    fig.text(0.02, 0.885, textwrap.fill(subtitle, 125), fontsize=9.5, color=INK2, va="top")
    ax = fig.add_axes([0.03, 0.78 - height, 0.94, height])
    ax.axis("off")
    cut = lambda s: s if len(s) <= 30 else s[:27] + "..."
    cells = [[cut("" if pd.isna(v) else str(v)) for v in row] for row in df_.itertuples(index=False)]
    t = ax.table(cellText=cells, colLabels=list(df_.columns), loc="upper left", cellLoc="left", colLoc="left", bbox=[0, 0, 1, 1])
    t.auto_set_font_size(False)
    t.set_fontsize(9.5)
    width = [max([len(str(c))] + [len(r[i]) for r in cells]) + 2 for i, c in enumerate(df_.columns)]
    for (r, c), cell in t.get_celld().items():
        cell.set_width(width[c] / sum(width))
        cell.set_edgecolor(GRID)
        if r == 0:
            cell.set_facecolor(HEAD); cell.get_text().set_color("white"); cell.get_text().set_fontweight("bold")
        else:
            cell.set_facecolor("white" if r % 2 else BAND); cell.get_text().set_family("monospace")
    finish(fig, name)

In [ ]:
#@title 7.2 What the raw files look like (header and first real rows)
table_picture(ip_head, "Raw file: item_properties_part1.csv",
              f"{n_rows['item_properties_part1.csv']:,} rows x 4 columns. Long format: 1 row = 1 property value of 1 item at 1 time. "
              "Most property names and values are coded.", "01_raw_item_properties_part1.png", 0.36)
table_picture(ip2_head, "Raw file: item_properties_part2.csv",
              f"{n_rows['item_properties_part2.csv']:,} rows x 4 columns, the same columns as part 1. Together: "
              f"{ip_rows:,} rows, {len(prop_rows):,} different property names.", "02_raw_item_properties_part2.png", 0.36)
ev_show = pd.read_csv(RAW_DIR / "events.csv", nrows=5, dtype=str)
table_picture(ev_show, "Raw file: events.csv", f"{n_events_raw:,} rows x 5 columns. 1 row = 1 customer action on 1 item. "
              "transactionid is empty unless the event is a purchase.", "03_raw_events.png", 0.36)
table_picture(pd.read_csv(RAW_DIR / "category_tree.csv", nrows=7, dtype=str), "Raw file: category_tree.csv",
              f"{len(tree):,} rows x 2 columns. 1 row = 1 category and its parent. An empty parentid = top-level category.",
              "04_raw_category_tree.png", 0.5)
show_cols = ["itemid", "snapshot", "split", "views_1w", "atc_1w", "tx_1w", "changes_4w", "cat", "parent_cat", "y_next_out"]
table_picture(df[show_cols].head(6), "Final table (input for the models)",
              f"{len(df):,} rows x {df.shape[1]} columns; 1 row = 1 in-stock, active item in 1 week. Showing {len(show_cols)} columns.",
              "05_final_table.png", 0.42)

In [ ]:
#@title 7.3 Data pipeline tree (drawn from the process log: every box is a table, every arrow is a step)
from matplotlib.patches import FancyBboxPatch

nodes, edges = {}, {}
for e in CLEAN_LOG:
    dst = e["output"]
    if not dst:
        continue
    n = nodes.setdefault(dst, {"rows": e["rows_after"], "cols": e["cols_after"], "inplace": [], "raw": not e["_src"]})
    n["rows"], n["cols"] = e["rows_after"], e["cols_after"]          # the table as it is after its last step
    for s in e["_src"]:
        if s == dst:
            n["inplace"].append(e["step"])
        else:
            edges.setdefault((s, dst), []).append(e["step"])
# layer = longest path from a raw file
layer = {k: 0 for k in nodes}
for _ in range(len(nodes)):
    for (a, b) in edges:
        layer[b] = max(layer[b], layer[a] + 1)
L = max(layer.values()) + 1
order = {}
for l in range(L):
    names = [k for k in nodes if layer[k] == l]
    if l:   # put a box under the mean position of the boxes that feed it
        names.sort(key=lambda k: np.mean([order[a] for (a, b) in edges if b == k and a in order] or [0]))
    for i, k in enumerate(names):
        order[k] = (i - (len(names) - 1) / 2) * 3.6
W, H = 3.0, 0.95
fig = plt.figure(figsize=(13, 1.25 * L + 1.4), dpi=110)
ax = fig.add_axes([0.01, 0.01, 0.98, 0.9])
ax.axis("off")
fig.text(0.02, 0.985, "Data pipeline: from the 4 raw files to the final table", fontsize=15, fontweight="bold", va="top")
fig.text(0.02, 0.957, textwrap.fill("Each box is a table (rows x columns after its last step). Each arrow is a processing step "
         "(cell number). Steps listed inside a box change that table in place. Dashed arrows bring a second input from higher up. "
         "Drawn from the process log of this run.", 140), fontsize=9.5, color=INK2, va="top")
pos = {k: (order[k], -layer[k] * 1.6) for k in nodes}
right = max(p[0] for p in pos.values()) + W / 2
lane = 0
for (a, b), steps in edges.items():
    (x1, y1), (x2, y2) = pos[a], pos[b]
    label = ", ".join(dict.fromkeys(steps))
    if layer[b] - layer[a] == 1:   # next level: a straight arrow
        ax.annotate("", xy=(x2, y2 + H / 2), xytext=(x1, y1 - H / 2),
                    arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1.2, shrinkA=0, shrinkB=0))
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, label, fontsize=8.5, ha="center", va="center", color=INK2,
                bbox=dict(boxstyle="round,pad=0.15", fc=SURFACE, ec="none"))
    else:                          # skips levels: go around on the right side, so no box is crossed
        lane += 1
        lx = right + 0.45 * lane
        yy = y2 + 0.18 - 0.12 * lane
        ax.plot([x1 + W / 2, lx, lx], [y1, y1, yy], color="#8a8980", lw=1.1, linestyle="--")
        ax.annotate("", xy=(x2 + W / 2, yy), xytext=(lx, yy),
                    arrowprops=dict(arrowstyle="-|>", color="#8a8980", lw=1.1, linestyle="--", shrinkA=0, shrinkB=0))
        ax.text(lx, (y1 + yy) / 2, label, fontsize=8.5, ha="center", va="center", color=INK2, rotation=90,
                bbox=dict(boxstyle="round,pad=0.15", fc=SURFACE, ec="none"))
for k, n in nodes.items():
    x, y = pos[k]
    final = k == "final table"
    fc = HEAD if n["raw"] else ("#c7556a" if final else "white")
    tc = "white" if (n["raw"] or final) else INK
    ax.add_patch(FancyBboxPatch((x - W / 2, y - H / 2), W, H, boxstyle="round,pad=0.02,rounding_size=0.08", fc=fc,
                                ec=GRID if fc == "white" else fc, lw=1.2))
    ax.text(x, y + 0.2, k, ha="center", va="center", fontsize=10, fontweight="bold", color=tc)
    ax.text(x, y - 0.08, f"{n['rows']:,} x {n['cols']}", ha="center", va="center", fontsize=9.5, color=tc, family="monospace")
    if n["inplace"]:
        ax.text(x, y - 0.33, "in place: " + ", ".join(dict.fromkeys(n["inplace"])), ha="center", va="center", fontsize=8, color=tc)
xs = [p[0] for p in pos.values()]
ax.set_xlim(min(xs) - W / 2 - 0.3, right + 0.45 * lane + 0.6)
ax.set_ylim(-(L - 1) * 1.6 - 0.9, 0.9)
fig.savefig(CHART_DIR / "06_pipeline_tree.png", dpi=160)
plt.show()

In [ ]:
#@title 7.4 Rows after each main step
fun = clean_log.dropna(subset=["funnel"])
fig, ax = figure("Rows after each main step",
                 "Item data from the raw file to the final table. The 3rd bar is larger than the 2nd because the format changes to one row "
                 "per item and week (the last status is carried forward); no data is added.", left=0.3)
y = np.arange(len(fun))[::-1]
ax.barh(y, fun["rows_after"], color=BLUE, height=0.55)
for yi, v in zip(y, fun["rows_after"]):
    ax.text(v + fun["rows_after"].max() * 0.01, yi, f"{v:,}", va="center")
ax.set_yticks(y, fun["funnel"])
ax.set_xlim(0, fun["rows_after"].max() * 1.15)
big = fun["rows_after"].max() >= 2e6
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v / 1e6:.0f}M" if big else f"{v / 1e3:.0f}K"))
ax.grid(axis="y", visible=False)
finish(fig, "07_rows_per_step.png")

In [ ]:
#@title 7.5 Target balance per week
tb = df.groupby("snapshot")["y_next_out"].mean().mul(100)
overall = 100 * df["y_next_out"].mean()
fig, ax = figure(f"Target balance: {overall:.1f}% of rows run out of stock the next week",
                 f"Share of rows with y_next_out = 1 per week ({len(tb)} weeks), with the time split.")
x = np.arange(len(tb))
for s in ["train", "validate", "test"]:
    idx = [i for i, w in enumerate(tb.index) if SPLIT[w] == s]
    ax.axvspan(idx[0] - 0.5, idx[-1] + 0.5, color=BAND if s != "validate" else SURFACE, zorder=0)
    ax.text(np.mean(idx), 0.97, s, transform=ax.get_xaxis_transform(), ha="center", va="top", color=INK2)
ax.bar(x, tb.values, color=BLUE, width=0.62)
ax.axhline(overall, color=INK2, linestyle="--", linewidth=1, label=f"all weeks {overall:.1f}%")
ax.legend(loc="upper right", bbox_to_anchor=(1, 0.9))
ax.set_ylim(0, tb.max() * 1.3)
ax.set_xticks(x, pd.to_datetime(tb.index).strftime("%d %b"), rotation=45, ha="right")
ax.set_ylabel("% of rows")
ax.grid(axis="x", visible=False)
finish(fig, "08_target_balance.png")

In [ ]:
#@title 7.6 Distribution of demand and the target rate per group
bins, names = [-1, 0, 1, 2, 5, 10, 20, 50, np.inf], ["0", "1", "2", "3-5", "6-10", "11-20", "21-50", "51+"]
g = df.assign(b=pd.cut(df["views_1w"], bins=bins, labels=names)).groupby("b", observed=False)["y_next_out"].agg(["size", "mean"])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), dpi=110, sharey=True)
fig.subplots_adjust(left=0.1, right=0.95, top=0.72, bottom=0.12, wspace=0.08)
fig.text(0.02, 0.955, "Views in the last 7 days, and the target rate in each group", fontsize=15, fontweight="bold", va="top")
fig.text(0.02, 0.885, f"All {len(df):,} rows. Median views {df['views_1w'].median():.0f}, max {df['views_1w'].max():,}.", fontsize=9.5, color=INK2, va="top")
y = np.arange(len(g))[::-1]
for a, vals, col, t in [(a1, 100 * g["size"] / g["size"].sum(), BLUE, "Share of rows (%)"), (a2, 100 * g["mean"], ORANGE, "Out of stock next week (%)")]:
    a.barh(y, vals, color=col, height=0.62)
    for yi, v in zip(y, vals):
        a.text(v + vals.max() * 0.02, yi, f"{v:.1f}", va="center", fontsize=9.5)
    a.set_title(t, loc="left", fontsize=11)
    a.set_xlim(0, vals.max() * 1.25)
    a.grid(axis="y", visible=False)
a1.set_yticks(y, names)
a1.set_ylabel("views in the last 7 days")
finish(fig, "09_views_distribution.png")

In [ ]:
#@title 7.7 Correlation of each feature with the target (train weeks only)
tr = df[df["split"] == "train"]
feats = [c for c in df.columns if c not in ("itemid", "snapshot", "split", "y_next_out", "cat", "parent_cat")]


def spearman(a, b):
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank())


corr = pd.Series({f: spearman(tr[f], tr["y_next_out"]) for f in feats}).dropna()
corr = corr.reindex(corr.abs().sort_values().index)
fig, ax = figure(f"Spearman correlation with the target (largest |r| = {corr.abs().max():.3f})",
                 f"Train weeks only ({len(tr):,} rows). Blue = positive, red = negative. Category codes are left out.", left=0.27)
y = np.arange(len(corr))
ax.barh(y, corr.values, color=[BLUE if v >= 0 else RED for v in corr.values], height=0.62)
off = corr.abs().max() * 0.03
for yi, v in zip(y, corr.values):
    ax.text(v + (off if v >= 0 else -off), yi, f"{v:+.3f}", va="center", ha="left" if v >= 0 else "right", fontsize=8.5)
ax.set_yticks(y, corr.index, fontsize=9)
lim = corr.abs().max() * 1.35
ax.set_xlim(-lim, lim)
ax.axvline(0, color=INK2, linewidth=0.8)
ax.grid(axis="y", visible=False)
finish(fig, "10_feature_correlation.png")

In [ ]:
#@title 7.8 Empty values per column of the final table
miss = (100 * df.isna().mean()).sort_values()
miss = miss[miss > 0]
fig, ax = figure("Empty values in the final table", "Share of rows with an empty value, per column (columns without empty values are not shown). "
                 f"Category 'Unknown' (a code, not empty): {100 * df['cat_missing'].mean():.1f}% of rows.", left=0.33)
y = np.arange(len(miss))
ax.barh(y, miss.values, color=BLUE, height=0.5)
for yi, v in zip(y, miss.values):
    ax.text(v + 1, yi, f"{v:.1f}%", va="center")
ax.set_yticks(y, miss.index, fontsize=10)
ax.set_xlim(0, 100)
ax.set_xlabel("% of rows")
ax.grid(axis="y", visible=False)
finish(fig, "11_empty_values.png")

## 8. Save everything

In [ ]:
#@title 8.1 Files: final dataset, data dictionary, cleaning log, quality checks, charts, and one zip
df.to_csv(OUT_DIR / "G31-04_Progress1_CleanedData.csv", index=False)
dictionary.to_csv(OUT_DIR / "data_dictionary.csv", index=False)
clean_log[LOG_COLS].to_csv(OUT_DIR / "cleaning_log.csv", index=False)
quality.to_csv(OUT_DIR / "data_quality_checks.csv", index=False)
with pd.ExcelWriter(OUT_DIR / "G31-04_Progress1_DataPrep.xlsx") as xw:
    for sheet, table in [("Cleaning_log", clean_log[LOG_COLS]), ("Data_quality", quality),
                         ("Data_dictionary", dictionary), ("Column_summary", summary.reset_index(names="column")),
                         ("First_rows", df.head(50))]:
        table.to_excel(xw, sheet_name=sheet, index=False)

zip_path = RUN_DIR / "G31-04_Progress1_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT_DIR.rglob("*")):
        if f.is_file():
            z.write(f, f.relative_to(RUN_DIR))
names = zipfile.ZipFile(zip_path).namelist()
print(f"{zip_path.name}: {zip_path.stat().st_size / 1e6:.1f} MB, {len(names)} files")
for n in names:
    print("  ", n)
if USE_MY_DRIVE and DRIVE_OUT.is_dir():
    shutil.copy(zip_path, DRIVE_OUT / zip_path.name)
    zipfile.ZipFile(zip_path).extractall(DRIVE_OUT)
    print("saved to", DRIVE_OUT)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))